# Week 43: Calibrated Abstention Extension

In [1]:
import csv
import json
import os
from pathlib import Path
from time import perf_counter

from ku_nlp_group_25_project.extensions.calibrated_abstention.calibrated_abstention_extension import CalibratedAbstentionExtension
from ku_nlp_group_25_project.extensions.calibrated_abstention.score_builder import ScoreBuilder
from ku_nlp_group_25_project.constants import LANGUAGES
from ku_nlp_group_25_project.utilities import load_dataset

SEED = 13

outputs_dir = Path.cwd().parent / "outputs"
outputs_sub_dir = outputs_dir / "week_43" / "calibrated_abstention_extension"
predictions_path = outputs_dir / "week_40_41" / "validation_predictions.csv"

outputs_sub_dir.mkdir(exist_ok=True, parents=True)

training_data, validation_data = load_dataset()
validation_data = validation_data.loc[
    validation_data["lang"].isin(LANGUAGES),
].reset_index(drop=True)

/home/kieran/code/github.com/kieranroneill/ku-nlp-group-25-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Fit the Decision Rules

In [2]:
with open(predictions_path, newline="", encoding="utf-8-sig") as handle:
    predictions = list(csv.DictReader(handle))

scores = ScoreBuilder().parse(predictions)
model = CalibratedAbstentionExtension(
    bootstrap_samples=2000,
    folds=5,
    seed=SEED,
    target_recall=0.90,
)
start = perf_counter()
result = model.evaluate(
    scores=scores,
)

print(f"Fitting time: {perf_counter() - start:.6f} seconds")

Fitting time: 1.485541 seconds


## 2. Results

In [3]:
with open(os.path.join(outputs_sub_dir, "metrics.csv"), "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["rule"] + list(next(iter(result.metrics.values())).to_dict()))
    writer.writeheader()

    for rule, metrics in result.metrics.items():
        writer.writerow({"rule": rule, **metrics.to_dict()})

with open(os.path.join(outputs_sub_dir, "metrics_by_language.csv"), "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["rule", "language"] + list(next(iter(next(iter(result.metrics_by_language.values())).values())).to_dict())
    )
    writer.writeheader()

    for rule, per_language in result.metrics_by_language.items():
        for language, metrics in per_language.items():
            writer.writerow({"rule": rule, "language": language, **metrics.to_dict()})

with open(os.path.join(outputs_sub_dir, "thresholds.json"), "w", encoding="utf-8") as handle:
    json.dump(
        {rule: thresholds.to_dict() for rule, thresholds in result.thresholds.items()},
        handle,
        indent=2,
    )

with open(os.path.join(outputs_sub_dir, "calibration.json"), "w", encoding="utf-8") as handle:
    json.dump(result.calibration.to_dict(), handle, indent=2)

with open(os.path.join(outputs_sub_dir, "transitions.json"), "w", encoding="utf-8") as handle:
    json.dump(
        {rule: transitions.to_dict() for rule, transitions in result.transitions.items()},
        handle,
        indent=2,
    )

with open(os.path.join(outputs_sub_dir, "summary.json"), "w", encoding="utf-8") as handle:
    json.dump(result.to_dict(), handle, indent=2)

### 2.1. Overall Results

In [4]:
for rule, metrics in result.metrics.items():
    print(
        f"{rule:26s} macro-F1 {metrics.macro_f1:.3f} | "
        f"F1 ans {metrics.answerable_f1:.3f} | F1 unans {metrics.unanswerable_f1:.3f} | "
        f"EM {metrics.exact_match:.3f} | F1 {metrics.token_f1:.3f} | "
        f"coverage {metrics.answerable_coverage:.3f} | abstention recall {metrics.abstention_recall:.3f}"
    )

print("\nFitted cutoffs:")

for rule, thresholds in result.thresholds.items():
    print(f"{rule:26s} cutoff {thresholds.cutoff:+.3f} temperature {thresholds.temperature:.3f}")

    for language, cutoff in thresholds.language_cutoffs.items():
        print(f"{language:>28s} cutoff {cutoff:+.3f}")

A_global_threshold         macro-F1 0.858 | F1 ans 0.963 | F1 unans 0.752 | EM 0.553 | F1 0.646 | coverage 0.975 | abstention recall 0.695
B_per_language_thresholds  macro-F1 0.867 | F1 ans 0.964 | F1 unans 0.771 | EM 0.558 | F1 0.652 | coverage 0.971 | abstention recall 0.738
C_temperature_scaled       macro-F1 0.858 | F1 ans 0.963 | F1 unans 0.752 | EM 0.553 | F1 0.646 | coverage 0.975 | abstention recall 0.695
D_constrained_recall       macro-F1 0.708 | F1 ans 0.865 | F1 unans 0.551 | EM 0.519 | F1 0.591 | coverage 0.775 | abstention recall 0.896
E_feature_combiner         macro-F1 0.862 | F1 ans 0.962 | F1 unans 0.762 | EM 0.558 | F1 0.651 | coverage 0.969 | abstention recall 0.732
F_fixed_probability        macro-F1 0.842 | F1 ans 0.962 | F1 unans 0.722 | EM 0.542 | F1 0.638 | coverage 0.987 | abstention recall 0.610

Fitted cutoffs:
A_global_threshold         cutoff -2.786 temperature 1.000
B_per_language_thresholds  cutoff -2.786 temperature 1.000
                          ar cu

### 2.2. Per-Language Results

In [5]:
for rule in ("A_global_threshold", "B_per_language_thresholds"):
    print(f"\n{rule}")
    for language, metrics in result.metrics_by_language[rule].items():
        print(
            f"  {language}: n {metrics.count} | macro-F1 {metrics.macro_f1:.3f} | "
            f"F1 ans {metrics.answerable_f1:.3f} | F1 unans {metrics.unanswerable_f1:.3f} | "
            f"coverage {metrics.answerable_coverage:.3f} | abstention recall {metrics.abstention_recall:.3f}"
        )


A_global_threshold
  ar: n 415 | macro-F1 0.937 | F1 ans 0.983 | F1 unans 0.891 | coverage 0.975 | abstention recall 0.942
  ko: n 356 | macro-F1 0.833 | F1 ans 0.982 | F1 unans 0.684 | coverage 0.982 | abstention recall 0.684
  te: n 384 | macro-F1 0.794 | F1 ans 0.917 | F1 unans 0.671 | coverage 0.966 | abstention recall 0.559

B_per_language_thresholds
  ar: n 415 | macro-F1 0.943 | F1 ans 0.985 | F1 unans 0.901 | coverage 0.975 | abstention recall 0.962
  ko: n 356 | macro-F1 0.869 | F1 ans 0.988 | F1 unans 0.750 | coverage 0.997 | abstention recall 0.632
  te: n 384 | macro-F1 0.801 | F1 ans 0.911 | F1 unans 0.690 | coverage 0.935 | abstention recall 0.634


### 2.3. Paired Transitions

In [6]:
for rule, transitions in result.transitions.items():
    print(
        f"{rule}: rescued abstentions {transitions.rescued_abstentions}, "
        f"broken abstentions {transitions.broken_abstentions}, "
        f"rescued answers {transitions.rescued_answers}, "
        f"broken answers {transitions.broken_answers}, unchanged {transitions.unchanged}"
    )

print(
    f"\nPaired macro-F1 difference (B - A): {result.comparison.difference:+.4f} "
    f"[{result.comparison.lower:+.4f}, {result.comparison.upper:+.4f}], "
    f"no improvement in {result.comparison.not_better_fraction:.1%} of replicates"
)

B_per_language_thresholds: rescued abstentions 8, broken abstentions 1, rescued answers 5, broken answers 9, unchanged 1132
D_constrained_recall: rescued abstentions 35, broken abstentions 2, rescued answers 2, broken answers 200, unchanged 916

Paired macro-F1 difference (B - A): +0.0098 [-0.0069, +0.0269], no improvement in 14.4% of replicates


### 2.4. Calibration Diagnostics

In [7]:
print(f"Temperature: {result.calibration.temperature:.3f}")
print(
    f"Expected calibration error: {result.calibration.expected_calibration_error:.4f} -> "
    f"{result.calibration.calibrated_expected_calibration_error:.4f}"
)
print(f"Average risk (AURC): {result.calibration.average_risk:.4f}")

for language, average_risk in result.calibration.average_risk_by_language.items():
    print(f"  {language}: {average_risk:.4f}")

print(f"Feature combiner AUC: {result.calibration.feature_area_under_roc:.3f}")
print(
    "\nMacro-F1 of the pooled and temperature-scaled rules: "
    f"{result.metrics['A_global_threshold'].macro_f1:.4f} vs "
    f"{result.metrics['C_temperature_scaled'].macro_f1:.4f}"
)

Temperature: 5.076
Expected calibration error: 0.0623 -> 0.0319
Average risk (AURC): 0.3446
  ar: 0.3111
  ko: 0.2844
  te: 0.4819
Feature combiner AUC: 0.915

Macro-F1 of the pooled and temperature-scaled rules: 0.8576 vs 0.8576


### 2.5. Error Inspection

In [8]:
baseline = result.decisions["A_global_threshold"]
candidate = result.decisions["B_per_language_thresholds"]

rescued = [
    score for score, was_answered, is_answered in zip(scores, baseline, candidate)
    if was_answered and not is_answered and not score.answerable
]
broken = [
    score for score, was_answered, is_answered in zip(scores, baseline, candidate)
    if not was_answered and is_answered and score.answerable
]

print("Rescued abstentions:")
for score in rescued[:5]:
    print(f"  {score.language} margin {score.margin:+.3f} span {score.predicted_span[:60]!r}")

print("\nBroken abstentions:")
for score in broken[:5]:
    print(f"  {score.language} margin {score.margin:+.3f} gold {score.gold_answer[:60]!r}")

Rescued abstentions:
  ar margin -3.219 span 'The Constitution of Kuwait supports the existence of politic'
  te margin -3.898 span '468.9 square miles'
  te margin -4.205 span 'Indonesia (210 million) and Pakistan'
  te margin -4.431 span 'Gabriel La Landelle'
  te margin -4.342 span 'Nigeria'

Broken abstentions:
  ko margin -0.639 gold 'Western Reserve University (formerly Western Reserve College'
  ko margin -1.767 gold 'Indonesian law does not recognise same-sex marriage'
  ko margin -2.309 gold 'part of Europe, Australasia, and the Americas'
  ko margin -1.132 gold 'Indonesia does not recognize same-sex marriage'
  ko margin -2.257 gold '10-billion'
